In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import glm
import seaborn as sns
import matplotlib.pyplot as plt
import os
import scanpy.experimental as sce 
import scipy
from scipy import sparse
import re
# ------------------------------------------------------------------------------
def load_and_extract_donor_meta(h5ad_path):
    """
    读取H5AD，提取Donor级别的核心信息（去重，避免细胞级重复）
    """
    # 读取H5AD文件
    adata = sc.read_h5ad(h5ad_path)
    print(f"✅ 成功读取H5AD文件，数据维度：{adata.shape}（细胞数 × 基因数）")
    
    # 定义需要提取的核心列（根据你的注释列名确认，无需修改）
    core_cols = ["donor_id", "sex", "development_stage"]
    
    # 检查核心列是否存在（避免列名错误）
    missing_cols = [col for col in core_cols if col not in adata.obs.columns]
    if missing_cols:
        raise ValueError(f"❌ H5AD的obs中缺失核心列：{missing_cols}")
    
    # 提取核心列并去重（每个Donor仅保留1条记录，避免细胞级重复）
    donor_meta = adata.obs[core_cols].drop_duplicates(subset="donor_id").reset_index(drop=True)
    
    print(f"\n📊 原始Donor统计：")
    print(f"- 总Donor数：{len(donor_meta)}")
    print(f"- 性别分布：")
    print(donor_meta["sex"].value_counts())
    print(f"- 年龄阶段数量：{donor_meta['development_stage'].nunique()} 个（19-75岁）")
    
    return donor_meta
# 2. 数据清洗：标准化性别和年龄格式
# ------------------------------------------------------------------------------
def clean_donor_meta(donor_meta):
    """
    清洗数据：统一性别格式、提取纯年龄数值（便于后续排序）
    """
    # 1. 标准化性别：仅保留“男性”相关值，过滤女性/未知性别
    # 常见男性标签（根据你的数据调整，如数据中是"M"/"Male"/"男"，这里已覆盖主流格式）
    male_keywords = {"male"}  # 若数据中男性用其他标签，需添加到集合中
    donor_meta["sex_clean"] = donor_meta["sex"].str.strip().str.lower()  # 去空格+转小写
    
    # 筛选男性Donor（仅保留男性数据）
    male_donor_meta = donor_meta[donor_meta["sex_clean"].isin([k.lower() for k in male_keywords])].copy()
    
    if len(male_donor_meta) == 0:
        raise ValueError("❌ 未筛选到男性Donor，请检查'sex'列的nv性标签是否在male_keywords中")
    
    # 2. 提取年龄数值（从"19-year-old stage"中提取纯数字19）
    def extract_age(age_stage):
        # 用正则匹配字符串中的数字（支持19、75等整数年龄）
        match = re.search(r"(\d+)-year-old", str(age_stage))
        return int(match.group(1)) if match else None
    
    male_donor_meta["age"] = male_donor_meta["development_stage"].apply(extract_age)
    
    # 删除年龄提取失败的记录（理论上你的数据不会出现，保险起见）
    male_donor_meta = male_donor_meta.dropna(subset=["age"]).reset_index(drop=True)
    
    # 按年龄升序排序（便于后续按年龄阶段分组查看）
    male_donor_meta = male_donor_meta.sort_values("age").reset_index(drop=True)
    
    print(f"\n🧹 清洗后nv性Donor统计：")
    print(f"- nv性Donor总数：{len(male_donor_meta)}")
    print(f"- 覆盖年龄阶段数：{male_donor_meta['development_stage'].nunique()} 个")
    
    return male_donor_meta
# 3. 按年龄阶段分组：生成每个阶段的男性Donor ID列表
# ------------------------------------------------------------------------------
def group_male_donor_by_age_stage(male_donor_meta):
    """
    按年龄阶段（如19-year-old stage）分组，汇总每个阶段的所有男性Donor ID
    """
    # 按年龄阶段分组，聚合Donor ID（用列表形式保存每个阶段的所有Donor）
    age_stage_donor = male_donor_meta.groupby("development_stage").agg(
        age=("age", "first"),  # 每个阶段的年龄（如19-year-old对应19）
        male_donor_count=("donor_id", "count"),  # 每个阶段的nv性Donor数量
        male_donor_ids=("donor_id", lambda x: list(x.unique()))  # 每个阶段的nv性Donor ID列表
    ).reset_index()
    
    # 按年龄升序排序（从19岁到75岁）
    age_stage_donor = age_stage_donor.sort_values("age").reset_index(drop=True)
    
    # 调整列顺序，便于阅读
    age_stage_donor = age_stage_donor[["development_stage", "age", "male_donor_count", "male_donor_ids"]]
    
    print(f"\n📈 各年龄阶段男性Donor分布（按年龄升序）：")
    print(age_stage_donor[["development_stage", "age", "male_donor_count"]].to_string(index=False))
    
    return age_stage_donor
# 4. 保存结果：生成Excel文件（便于查看）和DataFrame（便于后续分析）
# ------------------------------------------------------------------------------
def save_results(age_stage_donor, male_donor_meta, output_dir="."):
    """
    保存两个关键结果：
    1. 各年龄阶段的男性Donor汇总表（含ID列表）
    2. 清洗后的男性Donor完整元数据（可直接用于后续分析）
    """
    # 创建输出目录（若不存在）
    import os
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # 1. 保存各年龄阶段男性Donor汇总表（Excel）
    summary_path = os.path.join(output_dir, "male_donor_by_age_stage.xlsx")
    with pd.ExcelWriter(summary_path, engine="openpyxl") as writer:
        # 汇总表（含Donor ID列表）
        age_stage_donor.to_excel(writer, sheet_name="age_stage_male_donor", index=False)
        # 完整元数据表（每个男性Donor一行，含年龄/性别/Donor ID）
        male_donor_meta.to_excel(writer, sheet_name="male_donor_full_meta", index=False)
    
    print(f"\n💾 结果已保存至：{summary_path}")
    print(f"- 'age_stage_male_donor'工作表：各年龄阶段的nv性Donor数量和ID列表")
    print(f"- 'male_donor_full_meta'工作表：nv性Donor的完整元数据（可直接用于后续分析）")
    
    return summary_path
# 主函数：整合全流程
# ------------------------------------------------------------------------------
def main(h5ad_path, output_dir="."):
    # 1. 提取Donor核心元数据
    donor_meta = load_and_extract_donor_meta(h5ad_path)
    
    # 2. 清洗数据，筛选男性Donor并提取年龄
    male_donor_meta = clean_donor_meta(donor_meta)
    
    # 3. 按年龄阶段分组，汇总男性Donor ID
    age_stage_donor = group_male_donor_by_age_stage(male_donor_meta)
    
    # 4. 保存结果
    save_results(age_stage_donor, male_donor_meta, output_dir)
    
    # 返回可用于后续分析的DataFrame
    return age_stage_donor, male_donor_meta
# 运行代码（修改路径后直接执行）
# ------------------------------------------------------------------------------
if __name__ == "__main__":
    # 1. 修改为你的H5AD文件路径（必填）
    H5AD_PATH = "aidasnewblood.h5ad"  # 例如："sc_data.h5ad"
    
    # 2. 修改结果保存目录（可选，默认保存在当前文件夹）
    OUTPUT_DIR = "workdir/aidasfamale_donor_age_results"
    
    # 3. 执行全流程，获取结果
    age_stage_summary, male_donor_full = main(
        h5ad_path=H5AD_PATH,
        output_dir=OUTPUT_DIR
    )